In [1]:
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
from datasets import load_dataset
from peft import LoraConfig, get_peft_model

c:\study_6\nlp\lora_env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
W0330 11:30:25.980000 29792 Lib\site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


In [2]:
from datasets import load_dataset
from transformers import AutoTokenizer

# Load dataset
dataset = load_dataset("imdb")

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

# Tokenization function
def tokenize(example):
    return tokenizer(example["text"], truncation=True, padding="max_length")

# Apply tokenization
dataset = dataset.map(tokenize, batched=True)

# Rename label column
dataset = dataset.rename_column("label", "labels")

# Set format for PyTorch
dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])

# ✅ SMALL DATA (fast training)
train_dataset = dataset["train"].shuffle(seed=42).select(range(500))
test_dataset = dataset["test"].shuffle(seed=42).select(range(200))

Map: 100%|██████████| 50000/50000 [00:29<00:00, 1692.72 examples/s]


In [3]:
model_full = AutoModelForSequenceClassification.from_pretrained("bert-base-uncased", num_labels=2)

for param in model_full.parameters():
    param.requires_grad = True

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [4]:
training_args = TrainingArguments(
    output_dir="./results_full",
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=1,
    evaluation_strategy="epoch",
    logging_dir="./logs",
    save_strategy="no"
)

In [5]:
trainer_full = Trainer(
    model=model_full,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset
)

trainer_full.train()
full_results = trainer_full.evaluate()
print("Full Fine-tuning Results:", full_results)

  0%|          | 0/63 [00:00<?, ?it/s]c:\study_6\nlp\lora_env\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
                                               
100%|██████████| 63/63 [54:06<00:00, 51.54s/it]


{'eval_loss': 0.3985317349433899, 'eval_runtime': 241.5502, 'eval_samples_per_second': 0.828, 'eval_steps_per_second': 0.103, 'epoch': 1.0}
{'train_runtime': 3246.7782, 'train_samples_per_second': 0.154, 'train_steps_per_second': 0.019, 'train_loss': 0.5804189046223959, 'epoch': 1.0}


100%|██████████| 25/25 [03:16<00:00,  7.87s/it]

Full Fine-tuning Results: {'eval_loss': 0.3985317349433899, 'eval_runtime': 205.9802, 'eval_samples_per_second': 0.971, 'eval_steps_per_second': 0.121, 'epoch': 1.0}


In [6]:
model_lora = AutoModelForSequenceClassification.from_pretrained("bert-base-uncased", num_labels=2)

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["query", "value"],
    lora_dropout=0.1
)

model_lora = get_peft_model(model_lora, lora_config)

model_lora.print_trainable_parameters()

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


trainable params: 294,912 || all params: 109,778,690 || trainable%: 0.2686423020715587


In [9]:
trainer_lora = Trainer(
    model=model_lora,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset
)

trainer_lora.train()
lora_results = trainer_lora.evaluate()
print("LoRA Results:", lora_results)




























                                         
                                               
  6%|▋         | 4/63 [29:34<31:37, 32.15s/it]

100%|██████████| 63/63 [27:00<00:00, 25.72s/it]


{'eval_runtime': 168.2266, 'eval_samples_per_second': 1.189, 'eval_steps_per_second': 0.149, 'epoch': 1.0}
{'train_runtime': 1620.2125, 'train_samples_per_second': 0.309, 'train_steps_per_second': 0.039, 'train_loss': 0.702117193312872, 'epoch': 1.0}


100%|██████████| 25/25 [03:16<00:00,  7.84s/it]

LoRA Results: {'eval_runtime': 205.8069, 'eval_samples_per_second': 0.972, 'eval_steps_per_second': 0.121, 'epoch': 1.0}


In [10]:
import torch.nn as nn

class Adapter(nn.Module):
    def __init__(self, hidden_size):
        super().__init__()
        self.down = nn.Linear(hidden_size, hidden_size // 16)
        self.up = nn.Linear(hidden_size // 16, hidden_size)

    def forward(self, x):
        return x + self.up(torch.relu(self.down(x)))

In [11]:
for name, param in model_full.named_parameters():
    if "encoder.layer.10" in name or "encoder.layer.11" in name:
        param.requires_grad = True
    else:
        param.requires_grad = False

In [ ]:
print("Comparison:")
print("Full Fine-tuning:", full_results)
print("LoRA:", lora_results)

In [16]:


# Save model
trainer_full.save_model("./full_model")

# Save tokenizer (IMPORTANT)
tokenizer.save_pretrained("./full_model")





('./full_model\\tokenizer_config.json',
 './full_model\\special_tokens_map.json',
 './full_model\\vocab.txt',
 './full_model\\added_tokens.json',
 './full_model\\tokenizer.json')

In [17]:
# Save LoRA adapter (lightweight)
model_lora.save_pretrained("./lora_model")

# Save tokenizer
tokenizer.save_pretrained("./lora_model")

('./lora_model\\tokenizer_config.json',
 './lora_model\\special_tokens_map.json',
 './lora_model\\vocab.txt',
 './lora_model\\added_tokens.json',
 './lora_model\\tokenizer.json')

In [18]:
for name, param in model_full.named_parameters():
    if "attention" in name:
        param.requires_grad = True
    else:
        param.requires_grad = False

In [19]:
print("Comparison:")
print("Full Fine-tuning:", full_results)
print("LoRA:", lora_results)


Comparison:
Full Fine-tuning: {'eval_loss': 0.3985317349433899, 'eval_runtime': 205.9802, 'eval_samples_per_second': 0.971, 'eval_steps_per_second': 0.121, 'epoch': 1.0}
LoRA: {'eval_runtime': 205.8069, 'eval_samples_per_second': 0.972, 'eval_steps_per_second': 0.121, 'epoch': 1.0}
